# Week 7: QLoRA Fine-tuning Mistral-7B

**Runtime → Change runtime type → T4 GPU**

Upload train.json and validation.json via the Files panel (left sidebar folder icon).

In [5]:
!nvidia-smi

Wed Mar 25 15:32:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             31W /   70W |     919MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# STEP 1: Install dependencies (~3 minutes)
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 bitsandbytes==0.43.3 accelerate==0.34.2 datasets==3.0.0 huggingface_hub sentencepiece
print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [ ]:
# STEP 2: Login to HuggingFace
from huggingface_hub import login

HF_TOKEN = "your_huggingface_token_here"   # <- paste your HuggingFace WRITE token
HF_USERNAME = "your username"      # <- your HuggingFace username
MODEL_OUTPUT_NAME = "legal-mistral-7b"

login(token=HF_TOKEN)
print(f"Logged in! Model will be: {HF_USERNAME}/{MODEL_OUTPUT_NAME}")

Logged in! Model will be: hane123/legal-mistral-7b


In [8]:
# STEP 3: Upload training data (a file picker will appear below)
from google.colab import files
import json

print("Please upload train.json and validation.json when prompted...")
uploaded = files.upload()  # opens file picker — select both files

# Find the actual keys for train.json and validation.json, accounting for Colab's renaming
train_key = [k for k in uploaded.keys() if 'train' in k and '.json' in k][0]
val_key   = [k for k in uploaded.keys() if 'validation' in k and '.json' in k][0]

train_data = json.loads(uploaded[train_key].decode('utf-8'))
val_data   = json.loads(uploaded[val_key].decode('utf-8'))

print(f"Train: {len(train_data)} examples")
print(f"Val  : {len(val_data)} examples")
print(f"Sample: {train_data[0]['instruction'][:60]}")

Please upload train.json and validation.json when prompted...


Saving train.json to train (7).json
Saving validation.json to validation (7).json
Train: 799 examples
Val  : 100 examples
Sample: You are a legal expert. Answer the following question based 


In [9]:
# STEP 4: Convert to HuggingFace Dataset (Alpaca format)
from datasets import Dataset

def format_alpaca(example):
    return {
        "text": (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    }

train_dataset = Dataset.from_list([format_alpaca(ex) for ex in train_data])
val_dataset   = Dataset.from_list([format_alpaca(ex) for ex in val_data])

print(f"Train: {len(train_dataset)} rows")
print(f"Val  : {len(val_dataset)} rows")
print("Sample:")
print(train_dataset[0]["text"][:400])

Train: 799 rows
Val  : 100 rows
Sample:
### Instruction:
You are a legal expert. Answer the following question based ONLY on the provided legal document context. Be precise and faithful to the source.

### Input:
Context: . (B) Failure to accept offer.--If the contract holder does not accept the offer under paragraph (1) or if an agreement is not negotiated under paragraph (2)(D) within the time period described in subparagraph (A), the


***``Alpaca``*** format is just a standard way to organize training examples with 3 parts: what to do (instruction), what information is given (input), and what the correct answer is (output) — so Mistral-7B can learn from them!

In [ ]:
!pip show bitsandbytes

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

In [8]:
# Fix for CUDA 13.0 - install from source
!pip uninstall bitsandbytes -y -q
!pip install bitsandbytes --pre -q

In [10]:
# Compile bitsandbytes from source for CUDA 13.0
!pip uninstall bitsandbytes -y -q
!git clone https://github.com/bitsandbytes-foundation/bitsandbytes.git
%cd bitsandbytes
!cmake -DCOMPUTE_BACKEND=cuda -S .
!make
!pip install . -q
%cd /content
print("✅ bitsandbytes compiled from source!")

Cloning into 'bitsandbytes'...
remote: Enumerating objects: 14234, done.
remote: Counting objects: 100% (728/728), done.
remote: Compressing objects: 100% (297/297), done.
remote: Total 14234 (delta 609), reused 431 (delta 431), pack-reused 13506 (from 4)
Receiving objects: 100% (14234/14234), 5.81 MiB | 10.76 MiB/s, done.
Resolving deltas: 100% (9706/9706), done.
/content/bitsandbytes
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring bitsandbytes (Backend: cuda)
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - do

***``QLoRA``*** = a clever technique that makes fine-tuning a 7-billion parameter model possible on a free GPU by compressing the model to 4-bit AND only training tiny adapter layers instead of the whole model!

In [10]:
# STEP 5: Load Mistral-7B in 4-bit (QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os

# Set the BNB_CUDA_VERSION to explicitly use the compiled CUDA 12.8 library
os.environ["BNB_CUDA_VERSION"] = "128"

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit (~5 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print("Model loaded!")

Loading tokenizer...
Loading model in 4-bit (~5 minutes)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded!


In [11]:
# STEP 6: Configure LoRA adapters # LoRA : Low-Rank Adaptation
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA configured!")

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754
LoRA configured!


**Normal fine-tuning:** update ALL 7 billion parameters
                    → slow, expensive, needs huge GPU
                    
**LoRA fine-tuning:** freeze 99% of parameters
                  only train tiny "adapter" layers
                  → fast, cheap, same results! ✅

In [12]:
# STEP 7: Fine-tune with SFTTrainer (~2 hours on T4)
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none",
    evaluation_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    packing=False
)

print("Starting fine-tuning... (~2 hours on T4)")
trainer.train()
print("Fine-tuning complete!")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will over

Map:   0%|          | 0/799 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting fine-tuning... (~2 hours on T4)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


Fine-tuning complete!


In [14]:
# STEP 8: Push fine-tuned model to HuggingFace Hub
repo_id = f"{HF_USERNAME}/{MODEL_OUTPUT_NAME}"

print(f"Pushing to {repo_id}...")
trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print("Done! Copy this for your local notebook:")
print(f'  MODEL_ID = "{repo_id}"')

Pushing to hane123/legal-mistral-7b...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  554kB /  168MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pulyrt626/tokenizer.model: 100%|##########|  587kB /  587kB            

Done! Copy this for your local notebook:
  MODEL_ID = "hane123/legal-mistral-7b"
